In [1]:
# ===============================
# 1. Imports & Drive Mount
# ===============================
import pandas as pd
import numpy as np
from google.colab import drive

drive.mount('/content/drive')

# ===============================
# 2. Load CSV
# ===============================
file_path = "/content/drive/MyDrive/GEE_Forest_TimeSeries/AP_Forest_TimeSeries_Master.csv"
df = pd.read_csv(file_path)

features = ['NDVI', 'NDMI', 'NBR', 'BSI']
df = df[['latitude', 'longitude', 'year'] + features]
df = df.dropna()
df = df.sort_values(by=['latitude', 'longitude', 'year'])

# ===============================
# 3. Create Sequences (RAW DATA)
# ===============================
sequence_length = 4  # 2021–2024
X_raw = []
locations = []

for (lat, lon), group in df.groupby(['latitude', 'longitude']):
    if len(group) == sequence_length:
        X_raw.append(group[features].values)
        locations.append([lat, lon])

X_raw = np.array(X_raw)       # (samples, 4, 4)
locations = np.array(locations)

print("Raw input shape:", X_raw.shape)

# ===============================
# 4. Label Creation (BEFORE SCALING)
# ===============================
# NDVI drop = vegetation loss
ndvi_drop = X_raw[:, 0, 0] - X_raw[:, -1, 0]   # 2021 - 2024
bsi_rise  = X_raw[:, -1, 3] - X_raw[:, 0, 3]   # 2024 - 2021

ndvi_thr = np.percentile(ndvi_drop, 85)
bsi_thr  = np.percentile(bsi_rise, 85)

y = ((ndvi_drop > ndvi_thr) & (bsi_rise > bsi_thr)).astype(int)

print("\nLabel distribution:")
print("No deforestation:", (y == 0).sum())
print("Deforestation:", (y == 1).sum())

# ===============================
# 5. Train–Test Split (WITH LOCATIONS)
# ===============================
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test, loc_train, loc_test = train_test_split(
    X_raw, y, locations,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape :", X_test.shape)

# ===============================
# 6. Scaling (NO DATA LEAKAGE)
# ===============================
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

X_train_rs = X_train.reshape(-1, X_train.shape[-1])
X_test_rs  = X_test.reshape(-1, X_test.shape[-1])

X_train = scaler.fit_transform(X_train_rs).reshape(X_train.shape)
X_test  = scaler.transform(X_test_rs).reshape(X_test.shape)

print("Scaled train shape:", X_train.shape)

# ===============================
# 7. LSTM Model
# ===============================
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

model = Sequential([
    LSTM(64, input_shape=(X_train.shape[1], X_train.shape[2])),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        tf.keras.metrics.Recall(name='recall'),
        tf.keras.metrics.Precision(name='precision')
    ]
)

model.summary()

# ===============================
# 8. Train Model (Class Weighting)
# ===============================
class_weight = {0: 1, 1: 12}

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=20,
    batch_size=128,
    class_weight=class_weight,
    verbose=1
)

# ===============================
# 9. Evaluation
# ===============================
from sklearn.metrics import classification_report

y_prob = model.predict(X_test)
y_pred = (y_prob > 0.35).astype(int)   # recall-friendly threshold

print(classification_report(y_test, y_pred))

# ===============================
# 10. Save Predictions (CORRECT LOCATIONS)
# ===============================
results_df = pd.DataFrame({
    'latitude': loc_test[:, 0],
    'longitude': loc_test[:, 1],
    'deforestation': y_pred.flatten()
})

print(results_df.head())

results_df.to_csv(
    '/content/drive/MyDrive/AP_Deforestation_Predictions.csv',
    index=False
)

print("Saved:AP_Deforestation_Predictions.csv")


Mounted at /content/drive
Raw input shape: (30000, 4, 4)

Label distribution:
No deforestation: 26940
Deforestation: 3060
Train shape: (24000, 4, 4)
Test shape : (6000, 4, 4)
Scaled train shape: (24000, 4, 4)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 64)             │        17,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 19,777 (77.25 KB)

 Trainable params: 19,777 (77.25 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 7s 16ms/step - accuracy: 0.2693 - loss: 1.2992 - precision: 0.1199 - recall: 0.9433 - val_accuracy: 0.8172 - val_loss: 0.3981 - val_precision: 0.3535 - val_recall: 0.9559
Epoch 2/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.8500 - loss: 0.5398 - precision: 0.4042 - recall: 0.9340 - val_accuracy: 0.8533 - val_loss: 0.3158 - val_precision: 0.4073 - val_recall: 0.9624
Epoch 3/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.8721 - loss: 0.4515 - precision: 0.4338 - recall: 0.9475 - val_accuracy: 0.8992 - val_loss: 0.2202 - val_precision: 0.5031 - val_recall: 0.9379
Epoch 4/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.8963 - loss: 0.3749 - precision: 0.4991 - recall: 0.9644 - val_accuracy: 0.9120 - val_loss: 0.1972 - val_precision: 0.5380 - val_recall: 0.9706
Epoch 5/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.9147 - loss: 0.3100 - precision: 0.5461 - recall: 0.9710 - val_accuracy: 0.9043 - val_loss: 

In [2]:
!pip install folium
import pandas as pd
import folium
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd
import numpy as np

# 1️⃣ Load deforestation points
df_def = pd.read_csv('/content/drive/MyDrive/AP_Deforestation_Predictions.csv')
df_deforest = df_def[df_def['deforestation']==1]

# 2️⃣ Load time-series indices
df_ts = pd.read_csv('/content/drive/MyDrive/GEE_Forest_TimeSeries/AP_Forest_TimeSeries_Master.csv')

# 3️⃣ Sort and compute changes
df_ts_sorted = df_ts.sort_values(by=['latitude','longitude','year'])

# First and last year values per point
first_year = df_ts_sorted.groupby(['latitude','longitude']).first().reset_index()
last_year  = df_ts_sorted.groupby(['latitude','longitude']).last().reset_index()

df_change = pd.merge(first_year, last_year, on=['latitude','longitude'], suffixes=('_start','_end'))

# Compute delta indices
df_change['NDVI_change'] = df_change['NDVI_end'] - df_change['NDVI_start']
df_change['NBR_change']  = df_change['NBR_end']  - df_change['NBR_start']
df_change['BSI_change']  = df_change['BSI_end']  - df_change['BSI_start']
df_change['NDMI_change'] = df_change['NDMI_end'] - df_change['NDMI_start']


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# ==============================
# THRESHOLD FUNCTION
# ==============================
def get_thresholds(series):

    series = series.dropna()

    mean = series.mean()
    std  = series.std()
    q1   = series.quantile(0.25)
    q3   = series.quantile(0.75)

    return {
        'mean': mean,
        'std': std,
        'q1': q1,
        'q3': q3,
        'lower_2std': mean - 2*std,
        'upper_2std': mean + 2*std
    }

# ==============================
# COMPUTE THRESHOLDS
# ==============================
ndvi_thresh = get_thresholds(df_change['NDVI_change'])
nbr_thresh  = get_thresholds(df_change['NBR_change'])
bsi_thresh  = get_thresholds(df_change['BSI_change'])
ndmi_thresh = get_thresholds(df_change['NDMI_change'])

print("NDVI:", ndvi_thresh)
print("NBR :", nbr_thresh)
print("BSI :", bsi_thresh)
print("NDMI:", ndmi_thresh)

# ==============================
# MERGE DATA
# ==============================
df_merged = pd.merge(
    df_deforest,
    df_change[['latitude','longitude',
               'NDVI_change','NBR_change',
               'BSI_change','NDMI_change']],
    on=['latitude','longitude'],
    how='left'
)

# ==============================
# ADAPTIVE CLASSIFIER
# ==============================
def classify_cause_adaptive(row):

    ndvi = row['NDVI_change']
    nbr  = row['NBR_change']
    bsi  = row['BSI_change']

    # Missing data protection
    if pd.isna(ndvi) or pd.isna(nbr) or pd.isna(bsi):
        return "Unknown"

    # 🔥 Fire → extreme burn signal
    if nbr < nbr_thresh['lower_2std']:
        return "Fire"

    # 🪓 Logging → vegetation drop + soil exposure
    elif ndvi < ndvi_thresh['q1'] and bsi > bsi_thresh['q3']:
        return "Logging"

    # 🌾 Agriculture → moderate change
    elif ndvi < ndvi_thresh['mean'] and bsi > bsi_thresh['q1']:
        return "Agriculture"

    # 🏙 Urban / other disturbance
    else:
        return "Urban/Other"

# ==============================
# APPLY CLASSIFICATION
# ==============================
df_merged['cause'] = df_merged.apply(classify_cause_adaptive, axis=1)

# ==============================
# SAVE OUTPUT
# ==============================
df_merged.to_csv(
    '/content/drive/MyDrive/AP_Deforestation_Causes_Adaptive.csv',
    index=False
)

# ==============================
# SUMMARY
# ==============================
print("\nDeforestation Cause Summary:")
print(df_merged['cause'].value_counts())


NDVI: {'mean': np.float64(-0.19248976496061665), 'std': 0.12650220174030608, 'q1': np.float64(-0.27893157), 'q3': np.float64(-0.10388777500000002), 'lower_2std': np.float64(-0.4454941684412288), 'upper_2std': np.float64(0.0605146385199955)}
NBR : {'mean': np.float64(-0.15965633298339646), 'std': 0.15864833029569464, 'q1': np.float64(-0.27238727), 'q3': np.float64(-0.04804655499999998), 'lower_2std': np.float64(-0.4769529935747857), 'upper_2std': np.float64(0.15764032760799282)}
BSI : {'mean': np.float64(0.06998761816072377), 'std': 0.10791137935615212, 'q1': np.float64(-0.0067559943838463275), 'q3': np.float64(0.14808105833062546), 'lower_2std': np.float64(-0.14583514055158048), 'upper_2std': np.float64(0.285810376873028)}
NDMI: {'mean': np.float64(-0.09056214386234526), 'std': 0.12305318344464997, 'q1': np.float64(-0.1789936498), 'q3': np.float64(-0.003468771250000004), 'lower_2std': np.float64(-0.33666851075164517), 'upper_2std': np.float64(0.15554422302695467)}

Deforestation Cause 

In [4]:
import pandas as pd
import folium
from folium.plugins import MarkerCluster

# Andhra Pradesh map
# AP roughly: latitude 12.6–19.9, longitude 76.8–84.8
m = folium.Map(location=[16.5, 80.6], zoom_start=7)

# Load CSV
df = pd.read_csv('/content/drive/MyDrive/AP_Deforestation_Causes_Adaptive.csv')

print(df['cause'].value_counts())


cause
Logging        891
Fire           120
Agriculture     88
Name: count, dtype: int64


In [5]:
cause_colors = {
    'Fire': 'red',
    'Logging': 'orange',
    'Agriculture': 'yellow',
    'Urban/Other': 'gray'
}
# Cluster points for better performance
marker_cluster = MarkerCluster().add_to(m)

for _, row in df.iterrows():
    color = cause_colors.get(row['cause'], 'blue')

    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=3,
        color=color,
        fill=True,
        fill_opacity=0.7,
    ).add_to(marker_cluster)


In [6]:
m

In [7]:
m.save('/content/drive/MyDrive/AP_Deforestation_Causes_Adaptive_Map.html')


In [8]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
df_def = pd.read_csv(
    '/content/drive/MyDrive/AP_Deforestation_Predictions.csv'
)

print("Total samples:", len(df_def))
print(df_def.head())
df_deforest = df_def[df_def['deforestation'] == 1].copy()

print("Total deforestation samples:", len(df_deforest))
df_cause = pd.read_csv(
    '/content/drive/MyDrive/AP_Deforestation_Causes_Adaptive.csv'
)

print(df_cause.head())


Total samples: 6000
    latitude  longitude  deforestation
0  15.769476  80.278395              0
1  18.309911  79.432182              0
2  15.711085  79.754677              0
3  18.459930  79.467216              0
4  16.522264  80.180478              1
Total deforestation samples: 1099
    latitude  longitude  deforestation  NDVI_change  NBR_change  BSI_change  \
0  16.522264  80.180478              1    -0.308133   -0.263628    0.163087   
1  15.780255  80.281089              1    -0.279749   -0.369837    0.209050   
2  17.012744  80.408650              1    -0.337664   -0.312140    0.159994   
3  15.695814  79.693591              1    -0.606498   -0.639693    0.394417   
4  16.782775  80.192156              1    -0.341644   -0.287531    0.157067   

   NDMI_change    cause  
0    -0.180825  Logging  
1    -0.259918  Logging  
2    -0.195848  Logging  
3    -0.417477     Fire  
4    -0.155855  Logging  


In [9]:
import geopandas as gpd
from shapely.geometry import Point

# ==============================
# LOAD STATE DISTRICTS
# ==============================
ts = gpd.read_file('/content/drive/MyDrive/Telangana.geojson')
ap = gpd.read_file('/content/drive/MyDrive/AndhraPradesh.geojson')

# ==============================
# CRS SAFETY
# ==============================
for gdf in [ts, ap]:
    if gdf.crs is None:
        gdf.set_crs("EPSG:4326", inplace=True)

ts = ts.to_crs("EPSG:4326")
ap = ap.to_crs("EPSG:4326")

# ==============================
# ADD STATE LABEL (optional but useful)
# ==============================
ts["state"] = "Telangana"
ap["state"] = "Andhra Pradesh"

# ==============================
# COMBINE BOTH STATES
# ==============================
districts = gpd.GeoDataFrame(
    pd.concat([ts, ap], ignore_index=True),
    crs="EPSG:4326"
)

print(districts.head())


  REMARKS_2 Country State_Name State_Code     Dist_Name Dist_Code  \
0      None   India  Telangana       None        Nirmal      None   
1      None   India  Telangana       None     Hyderabad       536   
2      None   India  Telangana       None  Jaya Shankar      None   
3      None   India  Telangana       None    Kothagudem      None   
4      None   India  Telangana       None  Nagarkurnool      None   

                                            geometry      state  
0  POLYGON ((77.97691 19.32105, 77.9855 19.30899,...  Telangana  
1  POLYGON ((78.51785 17.51739, 78.5248 17.51434,...  Telangana  
2  POLYGON ((79.87325 18.82906, 79.88412 18.82813...  Telangana  
3  POLYGON ((80.48839 18.62672, 80.49812 18.61765...  Telangana  
4  POLYGON ((78.45483 16.97697, 78.46008 16.97691...  Telangana  


In [10]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point

# ==============================
# MERGE CAUSE DATA
# ==============================
df_deforest = df_deforest.merge(
    df_cause[['latitude', 'longitude', 'cause']],
    on=['latitude', 'longitude'],
    how='left'
)

print(df_deforest.head())

# ==============================
# CREATE POINT GEOMETRY
# ==============================
geometry = [
    Point(xy) for xy in zip(
        df_deforest['longitude'],
        df_deforest['latitude']
    )
]

gdf_points = gpd.GeoDataFrame(
    df_deforest,
    geometry=geometry,
    crs="EPSG:4326"
)

# ==============================
# LOAD TS + AP DISTRICTS
# ==============================
ts = gpd.read_file('/content/drive/MyDrive/Telangana.geojson')
ap = gpd.read_file('/content/drive/MyDrive/AndhraPradesh.geojson')

# CRS safety
for gdf in [ts, ap]:
    if gdf.crs is None:
        gdf.set_crs("EPSG:4326", inplace=True)

ts = ts.to_crs("EPSG:4326")
ap = ap.to_crs("EPSG:4326")

# optional labels
ts["state"] = "Telangana"
ap["state"] = "Andhra Pradesh"

# combine
districts = gpd.GeoDataFrame(
    pd.concat([ts, ap], ignore_index=True),
    crs="EPSG:4326"
)

print(districts.head())


    latitude  longitude  deforestation    cause
0  16.522264  80.180478              1  Logging
1  15.780255  80.281089              1  Logging
2  17.012744  80.408650              1  Logging
3  15.695814  79.693591              1     Fire
4  16.782775  80.192156              1  Logging
  REMARKS_2 Country State_Name State_Code     Dist_Name Dist_Code  \
0      None   India  Telangana       None        Nirmal      None   
1      None   India  Telangana       None     Hyderabad       536   
2      None   India  Telangana       None  Jaya Shankar      None   
3      None   India  Telangana       None    Kothagudem      None   
4      None   India  Telangana       None  Nagarkurnool      None   

                                            geometry      state  
0  POLYGON ((77.97691 19.32105, 77.9855 19.30899,...  Telangana  
1  POLYGON ((78.51785 17.51739, 78.5248 17.51434,...  Telangana  
2  POLYGON ((79.87325 18.82906, 79.88412 18.82813...  Telangana  
3  POLYGON ((80.48839 18.62672, 8

In [11]:
# ==============================
# SPATIAL JOIN
# ==============================
gdf_joined = gpd.sjoin(
    gdf_points,
    districts,
    how='left',
    predicate='intersects'
)

print(gdf_joined.head())

print("Total joined points:", len(gdf_joined))
print("Points without district:", gdf_joined['Dist_Name'].isna().sum())

print(gdf_joined['Dist_Name'].value_counts())

# ==============================
# FIX DUPLICATE CAUSE COLUMNS
# ==============================
if 'cause' not in gdf_joined.columns:

    if 'cause_x' in gdf_joined.columns and 'cause_y' in gdf_joined.columns:
        gdf_joined['cause'] = gdf_joined['cause_x'].fillna(gdf_joined['cause_y'])

    elif 'cause_x' in gdf_joined.columns:
        gdf_joined['cause'] = gdf_joined['cause_x']

    elif 'cause_y' in gdf_joined.columns:
        gdf_joined['cause'] = gdf_joined['cause_y']

# Optional cleanup
gdf_joined.drop(columns=[c for c in ['cause_x','cause_y'] if c in gdf_joined.columns],
                inplace=True)

# ==============================
# DISTRICT SUMMARY
# ==============================
district_summary = (
    gdf_joined
    .groupby('Dist_Name')
    .size()
    .reset_index(name='deforestation_points')
    .sort_values(by='deforestation_points', ascending=False)
)

print(district_summary)

print(
    "Sum of district-wise deforestation samples:",
    district_summary['deforestation_points'].sum()
)

district_summary.to_csv(
    '/content/drive/MyDrive/AP_District_Wise_Deforestation.csv',
    index=False
)

print("Saved district summary")

# ==============================
# DISTRICT × CAUSE SUMMARY
# ==============================
cause_summary = (
    gdf_joined
    .groupby(['Dist_Name', 'cause'])
    .size()
    .reset_index(name='count')
)

print(cause_summary.head())

cause_summary.to_csv(
    '/content/drive/MyDrive/AP_District_Wise_Deforestation_Causes.csv',
    index=False
)

print("Saved cause summary")


    latitude  longitude  deforestation    cause                   geometry  \
0  16.522264  80.180478              1  Logging  POINT (80.18048 16.52226)   
1  15.780255  80.281089              1  Logging  POINT (80.28109 15.78026)   
2  17.012744  80.408650              1  Logging  POINT (80.40865 17.01274)   
3  15.695814  79.693591              1     Fire  POINT (79.69359 15.69581)   
4  16.782775  80.192156              1  Logging  POINT (80.19216 16.78278)   

   index_right REMARKS_2 Country      State_Name State_Code Dist_Name  \
0           31      None   India  Andhra Pradesh         28    Guntur   
1           34      None   India  Andhra Pradesh         28  Prakasam   
2           32      None   India  Andhra Pradesh         28   Krishna   
3           34      None   India  Andhra Pradesh         28  Prakasam   
4           32      None   India  Andhra Pradesh         28   Krishna   

  Dist_Code           state  
0       548  Andhra Pradesh  
1       549  Andhra Pradesh  
2 

In [12]:
# =====================================================
# STEP 2: Load TS + AP District Boundaries
# =====================================================
ts = gpd.read_file('/content/drive/MyDrive/Telangana.geojson')
ap = gpd.read_file('/content/drive/MyDrive/AndhraPradesh.geojson')

for gdf in [ts, ap]:
    if gdf.crs is None:
        gdf.set_crs(epsg=4326, inplace=True)
    gdf.to_crs(epsg=4326, inplace=True)

ts["state"] = "Telangana"
ap["state"] = "Andhra Pradesh"

gdf_districts = gpd.GeoDataFrame(
    pd.concat([ts, ap], ignore_index=True),
    crs="EPSG:4326"
)


In [14]:
# =====================================================
# STEP 1: Imports
# =====================================================
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import folium
import numpy as np

# =====================================================
# STEP 2: Load Andhra Pradesh + Telangana Boundaries
# =====================================================

ap = gpd.read_file('/content/drive/MyDrive/AndhraPradesh.geojson')
ts = gpd.read_file('/content/drive/MyDrive/Telangana.geojson')

# CRS Safety
if ap.crs is None:
    ap.set_crs(epsg=4326, inplace=True)

if ts.crs is None:
    ts.set_crs(epsg=4326, inplace=True)

ap = ap.to_crs(epsg=4326)
ts = ts.to_crs(epsg=4326)

ap["state"] = "Andhra Pradesh"
ts["state"] = "Telangana"

# ✅ Merge both states (IMPORTANT)
gdf_districts = pd.concat([ap, ts], ignore_index=True)

print("Total districts loaded:", len(gdf_districts))

# =====================================================
# STEP 3: Load Deforestation Predictions
# =====================================================
df = pd.read_csv("/content/drive/MyDrive/AP_Deforestation_Predictions.csv")

if 'afforestation' not in df.columns:
    df['afforestation'] = 0

# =====================================================
# STEP 4: Convert to GeoDataFrame
# =====================================================
gdf_points = gpd.GeoDataFrame(
    df,
    geometry=[Point(xy) for xy in zip(df.longitude, df.latitude)],
    crs="EPSG:4326"
)

# =====================================================
# STEP 5: Spatial Join
# =====================================================
points_with_district = gpd.sjoin(
    gdf_points,
    gdf_districts,
    how="left",
    predicate="within"
)

# =====================================================
# STEP 6: District Statistics
# =====================================================
district_total = (
    points_with_district
    .groupby("Dist_Name")
    .size()
    .reset_index(name="total_samples")
)

district_deforestation = (
    points_with_district[points_with_district["deforestation"] == 1]
    .groupby("Dist_Name")
    .size()
    .reset_index(name="deforestation_samples")
)

district_afforestation = (
    points_with_district[points_with_district["afforestation"] == 1]
    .groupby("Dist_Name")
    .size()
    .reset_index(name="afforestation_samples")
)

# =====================================================
# STEP 7: Merge Stats
# =====================================================
gdf_districts = gdf_districts.merge(
    district_total, on="Dist_Name", how="left"
)
gdf_districts = gdf_districts.merge(
    district_deforestation, on="Dist_Name", how="left"
)
gdf_districts = gdf_districts.merge(
    district_afforestation, on="Dist_Name", how="left"
)

gdf_districts.fillna(0, inplace=True)

# =====================================================
# STEP 8: Area + Rate Calculations
# =====================================================
PIXEL_AREA_SQKM = 0.0001
PIXEL_AREA_HA = 0.01

gdf_districts["deforestation_rate_%"] = (
    gdf_districts["deforestation_samples"] /
    gdf_districts["total_samples"] * 100
).fillna(0)

gdf_districts["deforestation_area_sqkm"] = (
    gdf_districts["deforestation_samples"] * PIXEL_AREA_SQKM
)

gdf_districts["deforestation_area_ha"] = (
    gdf_districts["deforestation_samples"] * PIXEL_AREA_HA
)

gdf_districts["afforestation_rate_%"] = (
    gdf_districts["afforestation_samples"] /
    gdf_districts["total_samples"] * 100
).fillna(0)

gdf_districts["afforestation_area_sqkm"] = (
    gdf_districts["afforestation_samples"] * PIXEL_AREA_SQKM
)

gdf_districts["afforestation_area_ha"] = (
    gdf_districts["afforestation_samples"] * PIXEL_AREA_HA
)

# =====================================================
# STEP 9: Create Map (Centered Between AP & Telangana)
# =====================================================
m = folium.Map(location=[16.5, 79.5], zoom_start=7)

# =====================================================
# STEP 10: Choropleth
# =====================================================
min_val = gdf_districts["deforestation_samples"].min()
max_val = gdf_districts["deforestation_samples"].max()

if max_val == min_val:
    bins = [min_val, max_val + 1]
else:
    bins = np.linspace(min_val, max_val, num=6).tolist()

folium.Choropleth(
    geo_data=gdf_districts.to_json(),
    data=gdf_districts,
    columns=["Dist_Name", "deforestation_samples"],
    key_on="feature.properties.Dist_Name",
    fill_color="YlOrRd",
    bins=bins,
    fill_opacity=0.7,
    line_opacity=0.4,
    legend_name="Deforestation Samples (AP & Telangana)"
).add_to(m)

# =====================================================
# STEP 11: Tooltips
# =====================================================
folium.GeoJson(
    gdf_districts,
    tooltip=folium.GeoJsonTooltip(
        fields=[
            "Dist_Name",
            "state",
            "deforestation_samples",
            "deforestation_rate_%",
            "deforestation_area_sqkm",
            "deforestation_area_ha",
            "afforestation_samples",
            "afforestation_rate_%",
            "afforestation_area_sqkm",
            "afforestation_area_ha"
        ],
        aliases=[
            "District:",
            "State:",
            "Deforestation samples:",
            "Deforestation rate (%):",
            "Deforestation Area (sq.km):",
            "Deforestation Area (ha):",
            "Afforestation samples:",
            "Afforestation rate (%):",
            "Afforestation Area (sq.km):",
            "Afforestation Area (ha):"
        ],
        localize=True
    )
).add_to(m)

# =====================================================
# STEP 12: Combined State Alert Popup
# =====================================================
state_def = gdf_districts["deforestation_samples"].sum()

top3 = (
    gdf_districts.sort_values(by="deforestation_samples", ascending=False)
    .head(3)
)

top_districts_html = ""
for _, row in top3.iterrows():
    top_districts_html += f"• {row['Dist_Name']} ({int(row['deforestation_samples'])})<br>"

popup_html = f"""
<b>🌳 AP & Telangana Forest Alert</b><br><br>

Total Deforestation Points: <b>{int(state_def)}</b><br><br>

High Impact Districts:<br>
{top_districts_html}<br>

🌱 Recommended Actions:<br>
• Large-scale afforestation drives<br>
• Agroforestry promotion<br>
• Community forest monitoring<br>
• Sustainable land management
"""

folium.Marker(
    location=[16.5, 79.5],
    popup=popup_html,
    icon=folium.Icon(color="green")
).add_to(m)

# =====================================================
# STEP 13: Save Map
# =====================================================
folium.LayerControl().add_to(m)

m.save("/content/drive/MyDrive/ap_tel_forest.html")

print("✅ AP + Telangana forest map saved successfully!")

Total districts loaded: 40
✅ AP + Telangana forest map saved successfully!
